---

<div align="center">
  <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg" width="80"/>
</div>

<h1 align="center">GenAI & Advanced Nets: Bancos de Dados Vetoriais com FAISS</h1>

<h3 align="center">PhD. Julles Mitoura</h3>

<div align="center">
  <img src="https://img.shields.io/badge/Python-3776AB?style=for-the-badge&logo=python&logoColor=white"/>
  <img src="https://img.shields.io/badge/Jupyter-F37626?style=for-the-badge&logo=jupyter&logoColor=white"/>
  <img src="https://img.shields.io/badge/OpenAI-412991?style=for-the-badge&logo=openai&logoColor=white"/>
  <img src="https://img.shields.io/badge/Meta_AI-0082FB?style=for-the-badge&logo=meta&logoColor=white"/>
</div>

---

In [ ]:
# Obs: se você não estiver utilizando um ambiente virtual, instale as bibliotecas conforme se apresenta abaixo
#%pip install -q -r requirements.txt

# pip é o gerenciador de pacotes do Python. Pense nele como o instalador oficial de libs Python.
# no notebook, usar %pip ... é ideal porque instala no mesmo ambiente do kernel em uso.

# -q: quiet
# -r: requirement file, indica ao pip para instalar os pacotes listados no arquivo requirements.txt

# Biblioteca adicionada neste notebook:
# faiss-cpu — biblioteca do Meta AI para busca vetorial eficiente (CPU)
#             C++ core com Python bindings; suporta bilhões de vetores
#             instalar separadamente se necessário: pip install faiss-cpu

---

<div align="center">

## <span style="color:#1E90FF;">Visão Geral: de O(n) ao ANN</span>

</div>

Na aula anterior construímos um `VectorDatabase` em Python puro com **busca linear** — para cada consulta, calculamos a similaridade com todos os $n$ documentos indexados. Esse algoritmo tem complexidade **O(n)**: dobrar o número de documentos dobra o tempo de busca.

```
n = 100 docs   → busca em ~0.1 ms
n = 10.000     → busca em ~10 ms
n = 1.000.000  → busca em ~1.000 ms  (1 segundo por query!)
n = 1.000.000.000 → busca em ~16 min  (inviável em produção)
```

Para escalar RAG a corpora de milhões de documentos, precisamos de **Approximate Nearest Neighbor (ANN)** — algoritmos que trocam um pequeno grau de precisão por velocidade de busca ordens de magnitude maior, tipicamente **O(log n)** ou melhor.

### FAISS na Prática

**FAISS (Facebook AI Similarity Search)** é a biblioteca de referência para busca vetorial em escala. Desenvolvida pelo Meta AI Research, é usada internamente pelo Meta para indexar trilhões de embeddings. Neste notebook exploramos os três índices mais importantes:

| Índice | Busca | Treinamento | Quando usar |
|---|---|---|---|
| `IndexFlatL2` | Exata (L2) | Não | Corpora < 100k, precisão total |
| `IndexFlatIP` | Exata (inner product) | Não | Cosine similarity com vetores normalizados |
| `IndexIVFFlat` | Aproximada (IVF) | Sim | Corpora > 100k, velocidade prioritária |
| `IndexHNSWFlat` | Aproximada (grafo) | Não | Melhor custo-benefício em produção |

---

---

<div align="center">

## <span style="color:#1E90FF;">Setup: Configuração do Ambiente</span>

</div>

In [ ]:
import os
import re
import time
import numpy as np
import faiss
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
try:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    print("Cliente OpenAI configurado com sucesso.")
except Exception as e:
    print(f"Erro ao configurar o cliente OpenAI: {e}")
    exit(1)

print(f"FAISS versão: {faiss.__version__}")


# Função de embedding — mesma da Aula 7
def embed(text: str) -> list[float]:
    """Gera o embedding de um texto usando o modelo text-embedding-3-small."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding


# Frases de exemplo — mesmas da Aula 7 para comparação direta
frases = [
    "Gatos são animais domésticos muito populares.",
    "Felinos são conhecidos por sua independência.",
    "A inteligência artificial está revolucionando a tecnologia.",
    "Python é uma linguagem de programação versátil.",
    "Redes neurais são a base do deep learning.",
]

# Gerar e armazenar embeddings como array float32 (exigido pelo FAISS)
print("\nGerando embeddings das frases de exemplo...")
embeddings_list = [embed(f) for f in frases]
vectors = np.array(embeddings_list, dtype=np.float32)  # shape: (5, 1536)
DIM = vectors.shape[1]
print(f"Embeddings gerados: {vectors.shape}  (n_docs={vectors.shape[0]}, dim={DIM})")

---

<div align="center">

## <span style="color:#1E90FF;">IndexFlatL2: Busca Exata por Distância Euclidiana</span>

</div>

`IndexFlatL2` é o índice mais simples do FAISS: armazena todos os vetores e realiza **busca exata por força bruta**, calculando a distância L2 (Euclidiana) entre a query e cada vetor indexado.

A distância L2 ao quadrado entre dois vetores $\vec{u}$ e $\vec{v}$ é:

$$\|\vec{u} - \vec{v}\|^2 = \sum_{i=1}^{n} (u_i - v_i)^2$$

**Menor L2 = vetores mais próximos** (ao contrário da similaridade de cosseno, onde maior = mais similar).

### Características
- **Exato**: retorna sempre os k vizinhos corretos
- **Sem treinamento**: adicione vetores e busque imediatamente
- **Limitação**: O(n) — mesma complexidade do `VectorDatabase` da Aula 7, mas com implementação C++ muito mais rápida

---

In [ ]:
# --- IndexFlatL2 ---
index_flat_l2 = faiss.IndexFlatL2(DIM)

# add(): recebe array float32 de shape (n, d)
index_flat_l2.add(vectors)
print(f"Vetores indexados: {index_flat_l2.ntotal}")

# Query: mesma pergunta usada na Aula 7
query = "Qual é o comportamento de animais de estimação?"
query_vec = np.array([embed(query)], dtype=np.float32)  # shape: (1, 1536)

# search(): retorna (distâncias L2², índices) dos k mais próximos
k = 3
D, I = index_flat_l2.search(query_vec, k)

print(f'\nQuery: "{query}"\n')
print(f'{"Rank":<6} {"Idx":<6} {"Dist L2²":>10}  Texto')
print("-" * 75)
for rank, (idx, dist) in enumerate(zip(I[0], D[0]), start=1):
    print(f"  {rank:<4} {idx:<6} {dist:>10.4f}  {frases[idx]}")

print()
print("Nota: menor L2² = mais similar (inverso da similaridade de cosseno).")

---

<div align="center">

## <span style="color:#1E90FF;">Distâncias e Normalização: L2 vs Inner Product</span>

</div>

Na Aula 7 usamos **similaridade de cosseno** como métrica. O FAISS oferece L2 e **inner product** (produto interno). Existe uma relação matemática precisa entre elas para **vetores unitários** ($\|\vec{u}\| = \|\vec{v}\| = 1$):

$$\|\vec{u} - \vec{v}\|^2 = \|\vec{u}\|^2 - 2\langle\vec{u}, \vec{v}\rangle + \|\vec{v}\|^2 = 2 - 2\langle\vec{u}, \vec{v}\rangle = 2\bigl(1 - \cos(\vec{u}, \vec{v})\bigr)$$

Portanto, para vetores **normalizados** ($\|\vec{u}\| = \|\vec{v}\| = 1$):

$$\text{minimizar } \|\vec{u} - \vec{v}\|^2 \iff \text{maximizar } \langle\vec{u}, \vec{v}\rangle \iff \text{maximizar } \cos(\vec{u}, \vec{v})$$

**Conclusão prática**: se normalizarmos os vetores antes de indexar, `IndexFlatIP` (inner product) produz **exatamente os mesmos rankings** que busca por cosseno — mas com implementação FAISS otimizada.

A normalização é feita in-place com `faiss.normalize_L2(array)`, que divide cada vetor pela sua norma L2:

$$\vec{u}_{\text{norm}} = \frac{\vec{u}}{\|\vec{u}\|_2}$$

---

In [ ]:
# --- IndexFlatIP com vetores normalizados = cosine similarity ---

# Copia dos vetores para não modificar o array original (normalize_L2 é in-place)
vectors_norm = vectors.copy()
faiss.normalize_L2(vectors_norm)  # normaliza in-place: ||v|| = 1 para todo v

index_flat_ip = faiss.IndexFlatIP(DIM)
index_flat_ip.add(vectors_norm)
print(f"Vetores indexados (normalizados): {index_flat_ip.ntotal}")

# Normaliza a query também
query_norm = query_vec.copy()
faiss.normalize_L2(query_norm)

D_ip, I_ip = index_flat_ip.search(query_norm, k)

print(f'\nQuery: "{query}"\n')
print(f'{"Rank":<6} {"Idx":<6} {"Inner Prod":>12}  Texto')
print("-" * 75)
for rank, (idx, score) in enumerate(zip(I_ip[0], D_ip[0]), start=1):
    print(f"  {rank:<4} {idx:<6} {score:>12.4f}  {frases[idx]}")

print()
print("Nota: inner product de vetores normalizados = similaridade de cosseno.")
print("Os scores estão no intervalo [-1, 1] — maior = mais similar.")

# Verificação: ranking deve coincidir com IndexFlatL2 (ordem inversa de distância)
print(f"\nRanking FlatL2 : {list(I[0])}")
print(f"Ranking FlatIP : {list(I_ip[0])}")
print(f"Rankings iguais: {list(I[0]) == list(I_ip[0])}")

---

<div align="center">

## <span style="color:#1E90FF;">Busca Aproximada: por que o Flat não escala?</span>

</div>

`IndexFlatL2` e `IndexFlatIP` são exatos, mas percorrem **todos** os vetores a cada busca — O(n·d). Para 1 milhão de vetores com d=1536:

$$1{.}000{.}000 \times 1{.}536 \times 4 \text{ bytes} \approx 6 \text{ GB apenas para armazenamento}$$

E o custo de busca cresce linearmente com n. Para corpora maiores, a solução é aceitar um **trade-off precisão ↔ velocidade**:

| Estratégia | Ideia | Ganho |
|---|---|---|
| **IVF** (Inverted File) | Agrupa vetores em clusters; busca apenas nos clusters mais próximos | 10–100× mais rápido |
| **HNSW** (Hierarquical NSW) | Grafo de vizinhança hierárquico; navega pelos nós mais próximos | 100–1000× mais rápido |
| **PQ** (Product Quantization) | Comprime vetores reduzindo memória | 4–64× menos memória |

---

---

<div align="center">

## <span style="color:#1E90FF;">IndexIVFFlat: Busca por Arquivo Invertido</span>

</div>

`IndexIVFFlat` divide o espaço vetorial em **Voronoi cells** usando k-means:

```
Espaço vetorial particionado em nlist=4 clusters:

  ·  ·  ·  ·  |  ·  ·  ·
  ·  [C1] ·  |  · [C2] ·
  ·  ·  ·  ·  |  ·  ·  ·
  —————————————|————————————
  ·  ·  ·  ·  |  ·  ·  ·
  ·  [C3] ·  |  · [C4] ·
  ·  ·  ·  ·  |  ·  ·  ·

Query q → identifica cluster mais próximo → busca apenas nesse cluster
```

### Parâmetros

| Parâmetro | Papel | Regra prática |
|---|---|---|
| `nlist` | Número de clusters Voronoi | $\sqrt{n}$ a $4\sqrt{n}$ (ex: 100 para 10k docs) |
| `nprobe` | Clusters visitados por busca | Maior = mais preciso, mais lento; `nprobe=nlist` = exato |

**Atenção**: `IndexIVFFlat` exige uma etapa de **treinamento** (`index.train()`) que aprende os centroides dos clusters. Sem treinar, o índice levanta exceção.

---

In [ ]:
# --- IndexIVFFlat ---
# Usamos vetores sintéticos para demonstrar o treinamento
# (precisamos de pelo menos nlist vetores para treinar)

np.random.seed(42)
N_TRAIN = 500
D_DEMO = DIM  # mantemos a dimensão real dos embeddings

# Gera vetores sintéticos normalizados para o demo de IVFFlat
train_vecs = np.random.randn(N_TRAIN, D_DEMO).astype(np.float32)
faiss.normalize_L2(train_vecs)

nlist = 10  # número de clusters Voronoi

# O IVFFlat precisa de um 'quantizer' — índice interno que encontra o cluster mais próximo
quantizer = faiss.IndexFlatL2(D_DEMO)
index_ivf = faiss.IndexIVFFlat(quantizer, D_DEMO, nlist)

# PASSO OBRIGATÓRIO: treinamento (aprende os centroides via k-means)
print(f"Treinando IVFFlat com {N_TRAIN} vetores, nlist={nlist}...")
index_ivf.train(train_vecs)
print(f"Treinado: {index_ivf.is_trained}")

# Indexação
index_ivf.add(train_vecs)
print(f"Vetores indexados: {index_ivf.ntotal}")

# Query sintética
query_ivf = np.random.randn(1, D_DEMO).astype(np.float32)
faiss.normalize_L2(query_ivf)

print("\n=== Efeito do nprobe ===")
print(f"{' nprobe':<10} {'Resultados (top-5 índices)'}")
print("-" * 55)
for nprobe in [1, 3, 5, nlist]:
    index_ivf.nprobe = nprobe
    _, I_ivf = index_ivf.search(query_ivf, 5)
    print(f"  {nprobe:<8} {list(I_ivf[0])}")

print(f"\n  nprobe=nlist ({nlist}) → busca exata (mesmo resultado que FlatL2)")

In [ ]:
# --- Benchmark: FlatL2 vs IVFFlat ---
# Vetores sintéticos d=128 (mais rápido para gerar e indexar)

D_BM = 128
N_BM = 50_000
N_QUERIES = 100

np.random.seed(0)
bm_vecs   = np.random.randn(N_BM, D_BM).astype(np.float32)
bm_queries = np.random.randn(N_QUERIES, D_BM).astype(np.float32)

# --- FlatL2 ---
bm_flat = faiss.IndexFlatL2(D_BM)
bm_flat.add(bm_vecs)

t0 = time.perf_counter()
D_ref, I_ref = bm_flat.search(bm_queries, 5)
t_flat = time.perf_counter() - t0

# --- IVFFlat ---
nlist_bm = 100
quantizer_bm = faiss.IndexFlatL2(D_BM)
bm_ivf = faiss.IndexIVFFlat(quantizer_bm, D_BM, nlist_bm)
bm_ivf.train(bm_vecs)
bm_ivf.add(bm_vecs)
bm_ivf.nprobe = 5

t0 = time.perf_counter()
D_ivf, I_ivf = bm_ivf.search(bm_queries, 5)
t_ivf = time.perf_counter() - t0

# --- Recall: % de resultados corretos vs FlatL2 (ground truth) ---
recall = np.mean([
    len(set(I_ref[i]) & set(I_ivf[i])) / 5
    for i in range(N_QUERIES)
])

print("=" * 55)
print(f"Benchmark: {N_BM:,} vetores, d={D_BM}, {N_QUERIES} queries")
print("=" * 55)
print(f"{'Índice':<15} {'Tempo (s)':>12} {'Speedup':>10} {'Recall':>8}")
print("-" * 55)
print(f"{'FlatL2':<15} {t_flat:>12.4f} {'1.0x':>10} {'100%':>8}")
print(f"{'IVFFlat':<15} {t_ivf:>12.4f} {t_flat/t_ivf:>9.1f}x {recall*100:>7.1f}%")
print("=" * 55)
print(f"\nIVFFlat nlist={nlist_bm}, nprobe=5 — troca ~{(1-recall)*100:.0f}% de recall")
print(f"por um speedup de {t_flat/t_ivf:.1f}x em relação à busca exata.")

---

<div align="center">

## <span style="color:#1E90FF;">Persistência: Salvar e Recarregar Índices</span>

</div>

Em produção, a fase de indexação (embedding de todos os documentos) é executada **uma única vez** e o índice é salvo em disco. Nas consultas subsequentes, carregamos o índice diretamente — sem re-embetar nada.

O FAISS fornece duas funções simétricas:

```python
faiss.write_index(index, "caminho/para/arquivo.faiss")  # salva
index = faiss.read_index("caminho/para/arquivo.faiss")  # carrega
```

O arquivo `.faiss` é binário e contém tanto a estrutura do índice quanto todos os vetores armazenados. O carregamento é praticamente instantâneo mesmo para índices de gigabytes.

---

In [ ]:
# --- Persistência ---
INDEX_PATH = "dataset/rag_faiss.index"

# Salva o IndexFlatIP com as frases de exemplo
faiss.write_index(index_flat_ip, INDEX_PATH)
file_size_kb = os.path.getsize(INDEX_PATH) / 1024
print(f"Índice salvo em: {INDEX_PATH}  ({file_size_kb:.1f} KB)")

# Recarrega do disco
index_loaded = faiss.read_index(INDEX_PATH)
print(f"Índice carregado: {index_loaded.ntotal} vetores, d={index_loaded.d}")

# Verifica que o índice recarregado produz os mesmos resultados
D_loaded, I_loaded = index_loaded.search(query_norm, k)

print(f"\nResultados do índice original : {list(I_ip[0])}")
print(f"Resultados do índice carregado: {list(I_loaded[0])}")
print(f"Resultados idênticos: {list(I_ip[0]) == list(I_loaded[0])}")

---

<div align="center">

## <span style="color:#1E90FF;">RAG com FAISS</span>

</div>

Substituímos o `VectorDatabase` da Aula 7 por um `IndexFlatIP` com vetores normalizados. O pipeline permanece idêntico — apenas o componente de armazenamento e busca muda:

```
Aula 7:  chunks → embed() → VectorDatabase (listas + NumPy, O(n))
Aula 8:  chunks → embed() → IndexFlatIP   (FAISS C++, O(n) mas muito mais rápido)
```

Para corpora pequenos como um único artigo de 21 páginas, a diferença de velocidade é imperceptível — o ganho real aparece com dezenas de milhares de documentos. O objetivo aqui é demonstrar a **interface FAISS em um contexto RAG real** e mostrar a facilidade de substituição.

### Mapeamento de IDs

O FAISS armazena vetores como array NumPy e retorna **índices inteiros** (posição no array). Mantemos um mapeamento externo `lista de textos` para recuperar o conteúdo dos chunks:

```
FAISS index → retorna I = [42, 7, 15]  (posições)
chunk_texts[42], chunk_texts[7], chunk_texts[15]  → textos recuperados
```

---

In [ ]:
from pypdf import PdfReader

PDF_PATH = "docs/article.pdf"


# --- Funções de pré-processamento (mesmas da Aula 7) ---

def clean_text(text: str) -> str:
    """Remove artefatos de extração de PDF científico."""
    text = re.sub(r"Eng \d+, \d+, \d+\s*\n?\d* of \d+\n?", "", text)
    text = re.sub(r"-\n(\w)", r"\1", text)
    text = re.sub(r" {2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def chunk_text(text: str, chunk_size: int = 1000, overlap: int = 200) -> list[str]:
    """Divide texto em chunks com sobreposição."""
    chunks, step, start = [], chunk_size - overlap, 0
    while start < len(text):
        chunks.append(text[start : start + chunk_size])
        start += step
    return chunks


# --- Extração e chunking ---
reader = PdfReader(PDF_PATH)
raw_text = "\n".join(page.extract_text() or "" for page in reader.pages)
clean = clean_text(raw_text)
chunks = chunk_text(clean, chunk_size=1000, overlap=200)

print(f"PDF: {len(reader.pages)} páginas → {len(clean):,} chars limpos → {len(chunks)} chunks")

In [ ]:
# ============================================================
# FASE 1: INDEXAÇÃO COM FAISS
# ============================================================

print(f"Indexando {len(chunks)} chunks no FAISS...\n")

# Gerar embeddings de todos os chunks
chunk_embeddings = []
for i, chunk in enumerate(chunks):
    chunk_embeddings.append(embed(chunk))
    if (i + 1) % 10 == 0 or i == len(chunks) - 1:
        print(f"  Embeddings gerados: {i + 1}/{len(chunks)}")

# Montar array float32 e normalizar (para inner product = cosine)
chunk_vecs = np.array(chunk_embeddings, dtype=np.float32)
faiss.normalize_L2(chunk_vecs)

# Construir índice FAISS (IndexFlatIP com vetores normalizados = cosine search)
rag_index = faiss.IndexFlatIP(chunk_vecs.shape[1])
rag_index.add(chunk_vecs)

# Salvar índice em disco para reutilização futura
RAG_INDEX_PATH = "dataset/rag_faiss.index"
faiss.write_index(rag_index, RAG_INDEX_PATH)

print(f"\nIndexação concluída: {rag_index.ntotal} chunks no índice FAISS.")
print(f"Índice salvo em: {RAG_INDEX_PATH}")


# ============================================================
# FASE 2: CONSULTA RAG COM FAISS
# ============================================================

def rag_query_faiss(
    question: str,
    index: faiss.Index,
    texts: list[str],
    top_k: int = 3,
    verbose: bool = False
) -> str:
    """
    Consulta RAG usando FAISS como backend de busca vetorial.

    Args:
        question: Pergunta em linguagem natural.
        index: Índice FAISS indexado (IndexFlatIP com vetores normalizados).
        texts: Lista de textos correspondendo às posições do índice.
        top_k: Número de chunks a recuperar.
        verbose: Exibe chunks recuperados se True.

    Returns:
        Resposta gerada pelo gpt-4o-mini fundamentada nos chunks recuperados.
    """
    # Embed e normalizar a query
    q_vec = np.array([embed(question)], dtype=np.float32)
    faiss.normalize_L2(q_vec)

    # Busca FAISS — retorna (scores, índices inteiros)
    scores, indices = index.search(q_vec, top_k)

    resultados = [
        {"pos": int(idx), "text": texts[int(idx)], "score": float(score)}
        for idx, score in zip(indices[0], scores[0])
        if idx >= 0  # FAISS retorna -1 se não houver resultados suficientes
    ]

    if verbose:
        print(f"=== Chunks recuperados (top {top_k}) ===")
        for r in resultados:
            print(f"  [chunk_{r['pos']:03d}]  score={r['score']:.4f}")
            print(f"  {r['text'][:120].strip()}...")
            print()

    contexto = "\n\n---\n\n".join(r["text"] for r in resultados)

    system_prompt = (
        "Você é um assistente especializado em termodinâmica computacional e Python científico. "
        "Responda à pergunta do usuário com base EXCLUSIVAMENTE no contexto fornecido. "
        "Se a informação não estiver no contexto, informe que não encontrou no artigo. "
        "Seja preciso, técnico e cite dados numéricos quando disponíveis no contexto."
    )

    user_message = (
        f"Contexto extraído do artigo científico:\n\n{contexto}\n\n"
        f"Pergunta: {question}"
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.1,
    )

    return response.choices[0].message.content


# --- Demo: mesmas 3 perguntas da Aula 7 ---
perguntas = [
    "Quais equações de estado são suportadas pela biblioteca tes-thermo para modelar as não-idealidades do sistema reativo?",
    "Qual é a eficiência de gaseificação do etanol comparada à do metanol a 550°C e 25 MPa, de acordo com o artigo?",
    "Como a temperatura influencia a produção de hidrogênio e metano no processo de gaseificação supercrítica do etanol (SCWG)?",
]

for i, pergunta in enumerate(perguntas, start=1):
    print("=" * 72)
    print(f"PERGUNTA {i}: {pergunta}")
    print("=" * 72)
    resposta = rag_query_faiss(pergunta, rag_index, chunks, top_k=3, verbose=True)
    print(f"RESPOSTA:\n{resposta}")
    print()

---

<div align="center">

## <span style="color:#1E90FF;">Resumo: Comparativo de Índices e Próximos Passos</span>

</div>

### Comparativo: Aula 7 vs Aula 8

| Componente | Aula 7 — VectorDatabase | Aula 8 — FAISS |
|---|---|---|
| Implementação | Python puro + NumPy | C++ com Python bindings |
| Métrica | Cosseno (explícita) | L2 / Inner Product |
| Normalização | Manual | `faiss.normalize_L2()` |
| Busca | Linear O(n) | Exata ou aproximada |
| Persistência | Nenhuma (RAM) | `write_index` / `read_index` |
| Escalabilidade | Até ~10k docs | Bilhões de vetores |

### Guia de Escolha de Índice FAISS

| Cenário | Índice recomendado |
|---|---|
| < 10k docs, precisão total, protótipo | `IndexFlatIP` (normalizado) |
| 10k – 1M docs, velocidade importante | `IndexIVFFlat` (nlist≈√n, nprobe≈10) |
| > 1M docs, memória limitada | `IndexIVFPQ` (quantização de produto) |
| Produção geral, melhor custo-benefício | `IndexHNSWFlat` |

### O que foi construído nesta sequência

```
Aula 7 — RAG do zero:
  embed() → VectorDatabase (O(n)) → rag_query() → gpt-4o-mini

Aula 8 — RAG com FAISS:
  embed() → IndexFlatIP (C++, persistente) → rag_query_faiss() → gpt-4o-mini
```

O componente de geração (LLM) permanece idêntico nas duas aulas — o que muda é **onde e como os chunks são armazenados e recuperados**. Este é o ponto central da arquitetura RAG: os três estágios (Retrieve, Augment, Generate) são desacoplados e podem ser trocados independentemente.

---